# Examen Bimestral - Call Centre Dataset
**Nombre:** Gabriel Quilachamin 

**Curso:** Introducción a los Sistemas de Información  

**Fecha:** 2026-06-05

In [31]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

df = pd.read_excel('01 Call-Center-Dataset.xlsx')
df['Date'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date'].dt.month_name()

print(f'Total de registros : {len(df)}')
print(f'Columnas           : {df.columns.tolist()}')
print(f'\nValores nulos por columna:')
print(df.isnull().sum())
df.head()

Total de registros : 5000
Columnas           : ['Call Id', 'Agent', 'Date', 'Time', 'Topic', 'Answered (Y/N)', 'Resolved', 'Speed of answer in seconds', 'AvgTalkDuration', 'Satisfaction rating', 'Month']

Valores nulos por columna:
Call Id                         0
Agent                           0
Date                            0
Time                            0
Topic                           0
Answered (Y/N)                  0
Resolved                        0
Speed of answer in seconds    946
AvgTalkDuration               946
Satisfaction rating           946
Month                           0
dtype: int64


,Call Id,Agent,Date,Time,Topic,Answered (Y/N),Resolved,Speed of answer in seconds,AvgTalkDuration,Satisfaction rating,Month
0,ID0001,Diane,2021-01-01,09:12:58,Contract related,Y,Y,109.0,00:02:23,3.0,January
1,ID0002,Becky,2021-01-01,09:12:58,Technical Support,Y,N,70.0,00:04:02,3.0,January
2,ID0003,Stewart,2021-01-01,09:47:31,Contract related,Y,Y,10.0,00:02:11,3.0,January
3,ID0004,Greg,2021-01-01,09:47:31,Contract related,Y,Y,53.0,00:00:37,2.0,January
4,ID0005,Becky,2021-01-01,10:00:29,Payment related,Y,Y,95.0,00:01:00,3.0,January


---
## **KPI 1 – Tasa de Resolución en la Llamada**

In [32]:
total          = len(df)
contestadas    = (df['Answered (Y/N)'] == 'Y').sum()
no_contestadas = (df['Answered (Y/N)'] == 'N').sum()
resueltas      = (df['Resolved'] == 'Y').sum()
no_resueltas   = (df['Resolved'] == 'N').sum()

CR_total       = resueltas / total * 100
CR_contestadas = resueltas / contestadas * 100

print(f'Total de llamadas         : {total}')
print(f'Llamadas contestadas (Y)  : {contestadas}')
print(f'Llamadas no atendidas (N) : {no_contestadas}')
print(f'Llamadas resueltas (Y)    : {resueltas}')
print(f'Llamadas no resueltas (N) : {no_resueltas}')
print(f'---')
print(f'CR sobre total            : {CR_total:.2f}%')
print(f'CR sobre contestadas      : {CR_contestadas:.2f}%')

Total de llamadas         : 5000
Llamadas contestadas (Y)  : 4054
Llamadas no atendidas (N) : 946
Llamadas resueltas (Y)    : 3646
Llamadas no resueltas (N) : 1354
---
CR sobre total            : 72.92%
CR sobre contestadas      : 89.94%


**Interpretación de negocio:**  
El 89.94% de las llamadas atendidas fueron resueltas en el primer contacto, lo que evidencia una adecuada capacidad de resolución por parte de los agentes. Sin embargo, el 18.92% de llamadas no contestadas reduce el desempeño global del centro, por lo que disminuir el abandono es importante para mejorar este indicador

---
## **KPI 2 – Tiempo Promedio de Respuesta**

In [34]:

ASA = df['Speed of answer in seconds'].mean()  
percentiles = df['Speed of answer in seconds'].describe()

print(f'ASA Promedio       : {ASA:.2f} segundos ({ASA/60:.2f} minutos)')
print(f'Mínimo             : {percentiles["min"]:.0f} seg')
print(f'Máximo             : {percentiles["max"]:.0f} seg')
print(f'Mediana (P50)      : {percentiles["50%"]:.0f} seg')
print(f'P75                : {percentiles["75%"]:.0f} seg')
print(f'Base de cálculo    : {int(percentiles["count"])} llamadas contestadas')

ASA Promedio       : 67.52 segundos (1.13 minutos)
Mínimo             : 10 seg
Máximo             : 125 seg
Mediana (P50)      : 68 seg
P75                : 97 seg
Base de cálculo    : 4054 llamadas contestadas


**Interpretación de negocio:**  
El ASA promedio de 67.52 segundos supera el estándar de la industria de 60 segundos, indicando que la capacidad instalada del call center no cubre adecuadamente la demanda de llamadas entrantes. El agente Joe registra el mayor tiempo de respuesta promedio (70.99 seg), mientras que Becky presenta el más bajo (65.33 seg), lo que sugiere diferencias en la eficiencia individual de gestión de cola.

---
## **KPI 3 – Nivel de Satisfacción del Cliente (CSAT Promedio)**

In [36]:
CSAT = df['Satisfaction rating'].mean()
distribucion = df['Satisfaction rating'].value_counts().sort_index()
csat_por_tema = df.groupby('Topic')['Satisfaction rating'].mean().sort_values()

print(f'CSAT Promedio global: {CSAT:.2f} / 5.00')
print(f'Base de cálculo     : {df["Satisfaction rating"].count()} llamadas con rating')
print(f'\nDistribución de calificaciones:')
for rating, count in distribucion.items():
    pct = count / distribucion.sum() * 100
    print(f'  Calificación {int(rating)}: {count} llamadas ({pct:.1f}%)')
print(f'\nCSAT promedio por tema:')
print(csat_por_tema.round(2).to_string())   

CSAT Promedio global: 3.40 / 5.00
Base de cálculo     : 4054 llamadas con rating

Distribución de calificaciones:
  Calificación 1: 417 llamadas (10.3%)
  Calificación 2: 396 llamadas (9.8%)
  Calificación 3: 1218 llamadas (30.0%)
  Calificación 4: 1180 llamadas (29.1%)
  Calificación 5: 843 llamadas (20.8%)

CSAT promedio por tema:
Topic
Contract related     3.38
Payment related      3.40
Streaming            3.40
Technical Support    3.41
Admin Support        3.43


**Interpretación de negocio:**  
El CSAT promedio fue de 3.40 sobre 5, lo que refleja una satisfacción aceptable, pero con margen de mejora. Dado que las valoraciones son similares en todos los temas, sería conveniente enfocarse en mejorar la experiencia general del cliente, especialmente reduciendo los tiempos de espera y aumentando la resolución de casos.


---
## **KPI 4 – Volumen de Llamadas y Tasa de Abandono por Agente**

In [38]:
kpi4 = df.groupby('Agent').agg(
    Total_Llamadas = ('Call Id', 'count'),
    Contestadas    = ('Answered (Y/N)', lambda x: (x == 'Y').sum()),
    Perdidas       = ('Answered (Y/N)', lambda x: (x == 'N').sum()),
    Resueltas      = ('Resolved',       lambda x: (x == 'Y').sum()),
    CSAT_prom      = ('Satisfaction rating', 'mean'),
    ASA_prom       = ('Speed of answer in seconds', 'mean')
).reset_index()

kpi4['Tasa_Abandono_%']   = (kpi4['Perdidas']  / kpi4['Total_Llamadas'] * 100).round(2)
kpi4['Tasa_Resolucion_%'] = (kpi4['Resueltas'] / kpi4['Total_Llamadas'] * 100).round(2)
kpi4['CSAT_prom']         = kpi4['CSAT_prom'].round(2)
kpi4['ASA_prom']          = kpi4['ASA_prom'].round(2)

print(kpi4.to_string(index=False))

  Agent  Total_Llamadas  Contestadas  Perdidas  Resueltas  CSAT_prom  ASA_prom  Tasa_Abandono_%  Tasa_Resolucion_%
  Becky             631          517       114        462       3.37     65.33            18.07              73.22
    Dan             633          523       110        471       3.45     67.28            17.38              74.41
  Diane             633          501       132        452       3.41     66.27            20.85              71.41
   Greg             624          502       122        455       3.40     68.44            19.55              72.92
    Jim             666          536       130        485       3.39     66.34            19.52              72.82
    Joe             593          484       109        436       3.33     70.99            18.38              73.52
 Martha             638          514       124        461       3.47     69.49            19.44              72.26
Stewart             582          477       105        424       3.40     66.18  

**Interpretación de negocio:**  
Las llamadas estuvieron repartidas de forma similar entre los agentes. Sin embargo, algunos obtuvieron mejores resultados que otros, por lo que sería útil revisar las prácticas de los agentes con mejor desempeño para aplicarlas al resto del equipo.


---
## Resumen de KPIs

| KPI | Métrica | Valor obtenido | Referencia industria | Estado |
|-----|---------|---------------|----------------------|--------|
| KPI 1 – CR | Tasa de Resolución Global | 72.92% | ≥ 80% | Por debajo |
| KPI 1 – CR | CR sobre llamadas contestadas | 89.94% | ≥ 90% | Límite |
| KPI 2 – ASA | Tiempo promedio de respuesta | 67.52 seg | ≤ 60 seg | Por encima |
| KPI 3 – CSAT | Satisfacción promedio | 3.40 / 5.00 | ≥ 4.00 | Bajo |
| KPI 4 | Tasa de abandono más alta | Diane: 20.85% | ≤ 15% | Crítico |
| KPI 4 | Tasa de resolución más alta | Dan: 74.41% | ≥ 80% | Por debajo |

**Conclusión general:**  
Al analizar los cuatro KPIs se reveló que se opera por debajo de los estándares óptimos de la industria. El principal problema es el tiempo de espera elevado (ASA = 67.52 seg), que impulsa la tasa de abandono del 18.92% y deprime la satisfacción del cliente (CSAT = 3.40). Para revertir esta situación, se recomienda: 
(1) incrementar la capacidad de agentes en horas pico. 
(2) implementar un sistema de enrutamiento inteligente de llamadas.
(3) reforzar la capacitación de los agentes con mayor tasa de abandono, particularmente Diane.